In [1]:
!pip install llama-index-vector-stores-qdrant==0.3.0

  Using cached llama_index_core-0.11.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached sqlalchemy-2.0.48-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
  Using cached aiohttp-3.13.4-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached pillow-12.1.1-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached pydantic-2.1


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install llama-index==0.11.11

  Using cached llama_index-0.11.11-py3-none-any.whl.metadata (11 kB)
  Using cached llama_index_agent_openai-0.3.4-py3-none-any.whl.metadata (728 bytes)
  Using cached llama_index_cli-0.3.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached llama_index_embeddings_openai-0.2.5-py3-none-any.whl.metadata (686 bytes)
  Using cached llama_index_indices_managed_llama_cloud-0.11.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached llama_index_legacy-0.9.48.post4-py3-none-any.whl.metadata (8.5 kB)
  Using cached llama_index_llms_openai-0.2.16-py3-none-any.whl.metadata (3.3 kB)
  Using cached llama_index_multi_modal_llms_openai-0.2.3-py3-none-any.whl.metadata (729 bytes)
  Using cached llama_index_program_openai-0.2.0-py3-none-any.whl.metadata (766 bytes)
  Using cached llama_index_question_gen_openai-0.2.0-py3-none-any.whl.metadata (785 bytes)
  Using cached llama_index_readers_file-0.2.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached llama_index_readers_llama_parse-0.6.1-py3-none-any.whl.met


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from llama_index.core.schema import Document

In [24]:
from qdrant_client import QdrantClient
#client = QdrantClient(":memory:")
#client = QdrantClient(path="./qdrant_db")
client = QdrantClient("http://localhost", port=6333)

In [25]:
documents = [
    "고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.",
    "강아지는 충성심이 강하고 친절한 동물로, 흔히 인간의 최고의 친구로 불립니다. 주로 애완동물로 기르고, 동반자로서 유명합니다.",
    "고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다."
]
ids = ["doc1", "doc2", "doc3"]

documents = [Document(text=doc, id_=doc_id) for doc, doc_id in zip(documents, ids)]

In [26]:
#!pip install pywin32

In [28]:
collection_name = "llama_index_qdrant2"
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

True

In [29]:
#!pip install llama-index-embeddings-openai

In [30]:
# Qdrant 벡터 스토어 설정
vector_store = QdrantVectorStore(client=client, collection_name=collection_name)

# 인덱스 생성 및 저장
index = VectorStoreIndex.from_documents(documents, vector_store=vector_store)


In [31]:
# 쿼리 엔진 생성 및 실행
query_engine = index.as_query_engine()
query_text = "고양이"
response = query_engine.query(query_text)

print("[질의 결과]")
print(response)

[질의 결과]
고양이는 작은 육식동물로, 주로 애완동물로 기르며 민첩하고 장난기 있는 행동으로 유명합니다.


In [32]:
print("\n[응답에 사용된 문서]")
for i, node in enumerate(response.source_nodes, 1):
    print(f"{i}. {node.text}\n")


[응답에 사용된 문서]
1. 고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.

2. 고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다.

